# Olist E-commerce EDA — quantified findings
Brazilian marketplace, Sep 2016 – Oct 2018. Revenue basis: SUM(`order_items.price`) over **delivered** orders = **R$ 13,221,498.11**. Definitions: `METRICS.md`. Validation: `docs/EVAL.md`.

In [ ]:
import matplotlib
matplotlib.use('Agg')  # headless-safe; remove for interactive use
import matplotlib.pyplot as plt
import pandas as pd
RAW = '../data/raw/'
orders = pd.read_csv(RAW + 'olist_orders_dataset.csv', parse_dates=['order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'])
items = pd.read_csv(RAW + 'olist_order_items_dataset.csv')
pay = pd.read_csv(RAW + 'olist_order_payments_dataset.csv')
cust = pd.read_csv(RAW + 'olist_customers_dataset.csv')
prod = pd.read_csv(RAW + 'olist_products_dataset.csv')
trans = pd.read_csv(RAW + 'product_category_name_translation.csv')
rev = pd.read_csv(RAW + 'olist_order_reviews_dataset.csv')
print(orders.shape, items.shape, pay.shape, cust.shape)

## 1. Scale and fulfilment
- **99,441** orders; **96,478 delivered (97.0%)**; 625 cancelled, 609 unavailable.
- Window **2016-09-04 → 2018-10-17**; peak order month **2017-11 (7,544 orders)**.
- Funnel: 99,441 created → 99,281 approved → 97,658 with carrier → 96,476 delivered → 95,830 reviewed (99.3% of delivered leave a review).

In [ ]:
print(orders['order_status'].value_counts())
print('range:', orders['order_purchase_timestamp'].min(), '->', orders['order_purchase_timestamp'].max())
print('delivered share:', round((orders['order_status'] == 'delivered').mean(), 4))
for c in ['order_approved_at', 'order_delivered_carrier_date',
          'order_delivered_customer_date']:
    print(c, orders[c].notna().sum())

## 2. Revenue: two numbers, one decision
- Item revenue on delivered orders: **R$ 13,221,498.11** (the KPI).
- Payment total (all orders): **R$ 16,008,872.12** — higher because it bundles freight (≈R$ 2.25M) and instalment artefacts.
- AOV (delivered, items basis): **R$ 137.04**.

In [ ]:
d = orders[orders['order_status'] == 'delivered']
rev_items = items[items['order_id'].isin(set(d['order_id']))]['price'].sum()
print('delivered item revenue:', round(rev_items, 2))
print('payment_value total:', round(pay['payment_value'].sum(), 2))
print('freight total:', round(items['freight_value'].sum(), 2))
print('AOV:', round(rev_items / len(d), 2))

## 3. Seasonality — November peak, then a stable 2018 plateau
Best delivered-revenue month: **2017-11 (R$ 987,765)**; 2018-03/04/05 each deliver ≈R$ 950–978K on ~6.8–7.0K orders.

In [ ]:
d = d.copy()
d['month'] = d['order_purchase_timestamp'].dt.to_period('M').astype(str)
m = d.merge(items.groupby('order_id')['price'].sum().rename('rev'), on='order_id')
monthly = m.groupby('month').agg(orders=('order_id', 'nunique'), revenue=('rev', 'sum'))
print(monthly.sort_values('revenue', ascending=False).head(5).round(2))
monthly['revenue'].plot(kind='bar', figsize=(12, 4), title='Delivered revenue by month')
plt.tight_layout(); plt.show()

## 4. Categories — long tail, no single dominance
Top: **health_beauty (R$ 1.23M, 9.3%)**; watches_gifts R$ 1.17M; bed_bath_table R$ 1.02M. Highest AOV among leaders: watches_gifts (≈R$ 212).

In [ ]:
cat = prod[['product_id', 'product_category_name']].merge(trans, on='product_category_name', how='left')
cat['en'] = cat['product_category_name_english'].fillna(cat['product_category_name'])
di = items[items['order_id'].isin(set(d['order_id']))].merge(cat[['product_id', 'en']], on='product_id', how='left')
g = di.groupby('en').agg(revenue=('price', 'sum'), orders=('order_id', 'nunique'))
g['share'] = g['revenue'] / g['revenue'].sum()
print(g.sort_values('revenue', ascending=False).head(8).round(2))
g['revenue'].nlargest(10).plot(kind='barh', figsize=(8, 5), title='Top 10 categories by delivered revenue')
plt.tight_layout(); plt.show()

## 5. Retention — the honest headline: Olist is one-time purchase
- Only **3.0%** of delivered customers (2,801 / 93,358) ever place a second delivered order; month-1 cohort retention is typically **< 1%**.
- Implication: growth comes from acquisition and AOV, not repeat — retention work should target the 7,468 'At risk' RFM customers first.

In [ ]:
d2 = d.merge(cust[['customer_id', 'customer_unique_id']], on='customer_id')
d2['month'] = d2['order_purchase_timestamp'].dt.to_period('M').astype(str)
first = d2.groupby('customer_unique_id')['month'].min().rename('cohort')
u = d2[['customer_unique_id', 'month']].drop_duplicates().merge(first, on='customer_unique_id')
u['p'] = (u['month'].str[:4].astype(int) * 12 + u['month'].str[5:7].astype(int)
          - (u['cohort'].str[:4].astype(int) * 12 + u['cohort'].str[5:7].astype(int)))
ret = u.groupby(['cohort', 'p'])['customer_unique_id'].nunique().unstack(0)
ret = ret.div(ret.loc[0]).round(4)
print('month-1 retention for 2017 cohorts:')
print(ret.loc[1, ['2017-01', '2017-06', '2017-11']])
print('repeat-buyer rate:', round((first.reset_index().merge(
    d2.groupby('customer_unique_id')['order_id'].nunique().rename('n'),
    on='customer_unique_id')['n'] > 1).mean(), 4))

## 6. Customers — LTV is modest and right-skewed
Realised LTV per delivered customer: mean **R$ 141.62**, median **R$ 89.73**, p99 **R$ 1,004.99**. RFM split: Champions 14,908 · Loyal 22,435 · Recent 22,435 · At risk 7,468 · Lost 15,005.

In [ ]:
order_rev = items.groupby('order_id')['price'].sum().rename('orev')
ltv = d2.merge(order_rev, on='order_id', how='left').groupby(
    'customer_unique_id').agg(orders=('order_id', 'nunique'),
    monetary=('orev', 'sum'))
print(ltv['monetary'].describe().round(2))
print(ltv['monetary'].quantile([0.5, 0.9, 0.95, 0.99]).round(2))
ltv['monetary'].clip(upper=1000).hist(bins=30, figsize=(8, 4))
plt.title('Realised LTV per customer (capped at R$ 1,000)'); plt.show()

## 7. Quality — reviews are strong, lateness is the risk
Delivered avg review **4.16/5** (5★ 59.2%, 1★ 9.8%). **8.1%** of delivered orders arrive after the estimate — the prime suspect for low scores (join proposed in `docs/EVAL.md`, not yet tested). SP concentrates **38.3%** of revenue; credit_card is **79.2%** of payments.

In [ ]:
r = rev.merge(d[['order_id']], on='order_id')
print(r['review_score'].value_counts(normalize=True).sort_index().round(4))
late = (d['order_delivered_customer_date'] > d['order_estimated_delivery_date']).mean()
print('late rate:', round(float(late), 4))
# order-level modal payment type (same basis as dashboard + METRICS.md)
pm = pay.groupby('order_id').agg(pt=('payment_type',
    lambda s: s.mode().iat[0]), pv=('payment_value', 'sum'))
print('pay mix:', (pm.groupby('pt')['pv'].sum() / pm['pv'].sum()).round(3).to_dict())

## Recommendations (each traces to a number above)
1. **Protect November** (peak R$ 988K month): stock health_beauty / watches_gifts early; pre-book carrier capacity — fulfilment already dips 1.8% created→carrier.
2. **Attack the 8.1% late rate** before chasing acquisition: late delivery is the most plausible driver of the 9.8% 1-star reviews.
3. **Win back the 7,468 'At risk' customers** (high frequency, gone quiet) instead of generic reactivation — they proved repeat intent.
4. **Grow AOV, not just orders**: median LTV is R$ 89.73 vs mean R$ 141.62 — bundles in bed_bath_table / housewares (sub-R$ 110 AOV) lift the middle.